In [1]:
import scipy as sp
import numpy as np
from plotly import graph_objects as go
from plotly.subplots import make_subplots
from bayes_changepoint.stats import NormalInverseGamma, discrete_distribution
from bayes_changepoint.model import RunLengthModel

In [2]:
np.random.default_rng(seed=2)

Generator(PCG64) at 0x1E86EBDA880

In [3]:
# simulation parameters
n_state = 4
state_domain = [i_state for i_state in range(n_state)]
n_step = 500
inter_state_transition_prob = 0.01

transition_dist: list[discrete_distribution] = []

for i_state in range(n_state):
    transition_dist.append(
        discrete_distribution(state_domain, 
                              [1 - (n_state-1)*inter_state_transition_prob if j_state == i_state else inter_state_transition_prob for j_state in range(n_state)]))
    
init_dist = discrete_distribution(state_domain, [1/n_state for _ in range(n_state)])

# arbitrary values of the hyperparameters
mu_prior = 0.0
n_prior = 1
nu_prior = 10
sigma_prior = 10

In [4]:
# data simulation
# for now I choose some simple model with two states (two distinct sets of distribution parameters) 
# where the switching process will be modelled as a discrete markov chain





variance_prior = sp.stats.invgamma(a=nu_prior/2, scale=nu_prior*sigma_prior**2/2)
state_distributions = []

for i_state in range(n_state):
    variance = variance_prior.rvs()
    mean_prior = sp.stats.norm(loc=mu_prior, scale=np.sqrt(variance)/n_prior)
    mean = mean_prior.rvs()
    state_distributions.append(sp.stats.norm(loc=mean, scale=np.sqrt(variance)))
    print(f"Distribution {i_state}: mean={state_distributions[-1].mean()}, var={state_distributions[-1].var()}")


chain = []
chain.append(init_dist.sample())
samples = []
samples.append(state_distributions[chain[0]].rvs())

for i_step in range(1, n_step):
    chain.append(transition_dist[chain[i_step-1]].sample())
    samples.append(state_distributions[chain[i_step]].rvs())

Distribution 0: mean=-11.180730751119782, var=174.79673397823953
Distribution 1: mean=0.13458925578777386, var=100.51631296612867
Distribution 2: mean=4.5359377332076285, var=124.31876379022187
Distribution 3: mean=-19.726722251046098, var=135.020884795588


In [5]:
fig_obj = go.Figure()
fig_obj.add_trace(go.Scatter(y=chain, mode="markers+lines", name="True state"))
fig_obj.show()

fig_obj = go.Figure()
fig_obj.add_trace(go.Scatter(y=samples, mode="markers+lines", name="Samples"))
fig_obj.show()

In [6]:
# We start with the assumption of no history, i.e. only run length of zero is assumed.
run_length_model = RunLengthModel([0], 
                                  [NormalInverseGamma(mu_prior, sigma_prior, n_prior, mu_prior)],
                                  [1.0],
                                  NormalInverseGamma(mu_prior, sigma_prior, n_prior, mu_prior))

modus_lengths = []
modus_means = []
for i_step in range(n_step):
    if i_step % 100 == 0:
        print(f"Step={i_step}")
    run_length_model.grow(samples[i_step])
    modus_run_length = max(zip(run_length_model.run_length_distribution.domain, run_length_model.run_length_distribution.pmf), key=lambda item: item[1])
    modus_lengths.append(modus_run_length[0])
    modus_means.append(run_length_model.data_distributions[modus_run_length[0]].get_mean()[0])

Step=0


D:\github\BayesianOnlineChangepointDetection\src\bayes_changepoint\stats.py:98: RuntimeWarning:

divide by zero encountered in scalar divide



Step=100
Step=200
Step=300
Step=400


In [7]:
fig_obj = make_subplots(rows=3, cols=1, subplot_titles=["True state switching", "Modus length values", "Modus distribution mean"])
fig_obj.add_trace(go.Scatter(y=chain, mode="markers+lines", name="True state"), row=1, col=1)
fig_obj.add_trace(go.Scatter(y=modus_lengths, mode="markers+lines", name="Modus run length values"), row=2, col=1)
fig_obj.add_trace(go.Scatter(y=samples, mode="markers+lines", name="Samples"), row=3, col=1)
fig_obj.add_trace(go.Scatter(y=modus_means, mode="lines", name="Modus distribution mean"), row=3, col=1)
for i_state in range(n_state):
    fig_obj.add_hline(y=state_distributions[i_state].mean(), row=3, col=1)
fig_obj.show()